In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q18/q18_rewrite/checkpoints/pre_cell_3.pickle

In [4]:
%%RecordEvent
%%time
### cell 3 ###

# build a mapping from C_CUSTKEY to C_NAME on the GPU
cust_map = customer.set_index("C_CUSTKEY")["C_NAME"]

# 1) group and sum L_QUANTITY by order, 2) filter >300
gb1 = (
    lineitem
      .groupby("L_ORDERKEY", as_index=False, sort=False)["L_QUANTITY"]
      .sum()
)
gb1 = gb1[gb1.L_QUANTITY > 300]

# join with the needed orders columns
df = gb1.merge(
    orders[["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_TOTALPRICE"]],
    left_on="L_ORDERKEY", right_on="O_ORDERKEY"
)

# map in the customer name instead of a second merge
df["C_NAME"] = df["O_CUSTKEY"].map(cust_map)

# rename, select and sort
df = df.rename(columns={"O_CUSTKEY": "C_CUSTKEY"})
total = (
    df[["C_NAME", "C_CUSTKEY", "O_ORDERKEY", "O_ORDERDATE", "O_TOTALPRICE", "L_QUANTITY"]]
      .sort_values(["O_TOTALPRICE", "O_ORDERDATE"], ascending=[False, True])
)

CPU times: user 74.3 ms, sys: 51.5 ms, total: 126 ms
Wall time: 128 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q18/rewritten/o4_mini_high_small/checkpoints/post_cell_3_try_1.pickle